# AMMS 302 — Week 7: Advanced Database Design & Joins
**ER diagram · FOREIGN KEY · INNER/LEFT/RIGHT/FULL JOIN · SELF JOIN · dot notation · UNION · GROUP BY · HAVING**

> ต่อจาก `healthinfo.db` (สัปดาห์ 5–6) — รันออฟไลน์บน Windows ด้วย `uv run jupyter lab` · เปิดคู่กับ [สไลด์ wk07](./wk07.html)

### 🎯 Learning objectives (CLO1/CLO2)
- วาด ER diagram และประกาศ FOREIGN KEY ใน SQLite ได้
- เขียน JOIN ทุกชนิด + SELF JOIN + UNION ได้ถูกต้อง
- ใช้ GROUP BY / HAVING / dot notation เพื่อสรุปสถิติเชิงคลินิกได้

### 🗺️ แผนที่สไลด์ ↔ โน้ตบุ๊ก
| ipynb § | หัวข้อ | สไลด์ wk07 |
|---|---|---|
| §2 | FOREIGN KEY + PRAGMA | 03 |
| §3–5 | INNER/LEFT/(RIGHT/FULL) | 04–06 |
| §6 | SELF JOIN | 07 |
| §7 | UNION | 08 |
| §8–9 | GROUP BY / HAVING | 09–10 |
| §10 | Pipeline ฉบับเต็ม | 11 |

### 📚 Official references
- SQLite: [foreignkeys.html](https://www.sqlite.org/foreignkeys.html) · [lang_select.html#joinclause](https://www.sqlite.org/lang_select.html#joinclause) · [compound (UNION)](https://www.sqlite.org/lang_select.html#compound) · [GROUP BY](https://www.sqlite.org/lang_select.html#groupby) · [HAVING](https://www.sqlite.org/lang_select.html#having) · [PRAGMA foreign_keys](https://www.sqlite.org/pragma.html#pragma_foreign_keys)
- Python: [sqlite3](https://docs.python.org/3/library/sqlite3.html)
- W3Schools: [JOIN](https://www.w3schools.com/sql/sql_join.asp) · [INNER](https://www.w3schools.com/sql/sql_join_inner.asp) · [LEFT](https://www.w3schools.com/sql/sql_join_left.asp) · [RIGHT](https://www.w3schools.com/sql/sql_join_right.asp) · [FULL](https://www.w3schools.com/sql/sql_join_full.asp) · [GROUP BY](https://www.w3schools.com/sql/sql_groupby.asp) · [HAVING](https://www.w3schools.com/sql/sql_having.asp) · [UNION](https://www.w3schools.com/sql/sql_union.asp)
- Diagram tool: [draw.io](https://www.drawio.com/) (ER diagram ฟรี) · OMOP schema: [CommonDataModel](https://ohdsi.github.io/CommonDataModel/cdm54.html)

---


In [ ]:
# §0 เตรียม DB — ใช้ healthinfo.db ต่อ (ถ้าไม่มี รัน week05 notebook ก่อน)
import sqlite3, pathlib, pandas as pd
db = pathlib.Path("healthinfo.db")
assert db.exists(), "healthinfo.db not found — run week05-sql-basics.ipynb first"
con = sqlite3.connect(db)
cur = con.cursor()
print("patients rows:", cur.execute("SELECT COUNT(*) FROM patients").fetchone()[0])
print("sqlite version:", sqlite3.sqlite_version, "(RIGHT/FULL JOIN ต้อง ≥ 3.39)")

## §2 FOREIGN KEY — สายใยเชื่อมตาราง (สไลด์ 03)
สร้างตาราง `prescriptions` อ้างผู้ป่วยด้วย FK — SQLite **ปิด enforce FK โดย default** ต้อง `PRAGMA foreign_keys=ON` ต่อ connection

```
patients ──1:N──▶ prescriptions
(patient_id PK)   (rx_id PK, patient_id FK → patients)
```


In [ ]:
# §2.1 เปิด FK enforcement (ทำต่อ connection ใหม่ทุกครั้ง)
cur.execute("PRAGMA foreign_keys = ON")
print("fk:", cur.execute("PRAGMA foreign_keys").fetchone()[0], "(1=enforced)")

# §2.2 สร้าง prescriptions + seed ข้อมูล
cur.execute("DROP TABLE IF EXISTS prescriptions")
cur.execute("""
CREATE TABLE prescriptions(
  rx_id        INTEGER PRIMARY KEY,
  patient_id   INTEGER NOT NULL REFERENCES patients(patient_id),
  drug         TEXT NOT NULL,
  dose_mg      REAL,
  rx_date      TEXT,
  cost_thb     REAL
)""")
seed = [
 (1,10001,'Metformin',500,'2025-03-01',35.0),(2,10001,'Lisinopril',10,'2025-03-01',42.0),
 (3,10002,'Metformin',850,'2025-03-02',55.0),(4,10003,'Atorvastatin',20,'2025-03-03',68.0),
 (5,10003,'Metformin',500,'2025-04-01',35.0),(6,99999,'OrphanDrug',10,'2025-04-02',90.0)]
cur.executemany("INSERT INTO prescriptions VALUES (?,?,?,?,?,?)", seed)
con.commit()
print(pd.read_sql("SELECT * FROM prescriptions", con))

# §2.3 FK บล็อก orphan: ลองลบ patient 10001
try:
    cur.execute("DELETE FROM patients WHERE patient_id=10001"); con.commit()
    print("deleted (FK off?)")
except sqlite3.IntegrityError as e:
    print(f"✅ FK blocked: {e}")
    con.rollback()
print("patients still:", cur.execute("SELECT COUNT(*) FROM patients").fetchone()[0])

## §3 INNER JOIN — เฉพาะที่จับคู่ได้ (สไลด์ 04)
Venn: จุดตัด A∩B — แถวไหน FK ไม่ match จะหล่นทิ้ง


In [ ]:
# §3.1 INNER JOIN + dot notation + alias (สไลด์ 08)
q_inner = """
SELECT p.patient_id, p.hn, r.drug, r.dose_mg, r.cost_thb
FROM patients AS p                 -- alias p
INNER JOIN prescriptions AS r      -- alias r
        ON r.patient_id = p.patient_id
LIMIT 6"""
display(pd.read_sql(q_inner, con))

# §3.2 สังเกต: rx_id=6 (patient 99999 ไม่มีจริง) หายไป!
n_p = cur.execute("SELECT COUNT(*) FROM patients").fetchone()[0]
n_r = cur.execute("SELECT COUNT(*) FROM prescriptions").fetchone()[0]
n_i = cur.execute("SELECT COUNT(*) FROM patients p JOIN prescriptions r ON r.patient_id=p.patient_id").fetchone()[0]
print(f"patients={n_p}, prescriptions={n_r}, inner={n_i} ← rx_id=6 (orphan) หล่น")

## §4 LEFT JOIN — รักษาตารางซ้ายครบ (สไลด์ 05)
Venn: A ทั้งหมด + ส่วนตัด B — ไม่ match = NULL (คอลัมน์ฝั่งขวา)

💡 ในงานสาธารณสุขนิยม LEFT เพื่อไม่ทำผู้ป่วย "หาย" จากรายงาน


In [ ]:
# §4.1 LEFT JOIN จาก patients (ซ้าย)
q_left = """
SELECT p.patient_id, p.hn, r.drug
FROM patients p
LEFT JOIN prescriptions r ON r.patient_id = p.patient_id
ORDER BY p.patient_id LIMIT 8"""
display(pd.read_sql(q_left, con))

# §4.2 หาผู้ป่วย "ยังไม่เคยจ่ายยา" — pattern IS NULL after LEFT JOIN
q_no_rx = """
SELECT p.patient_id, p.hn
FROM patients p
LEFT JOIN prescriptions r ON r.patient_id = p.patient_id
WHERE r.rx_id IS NULL LIMIT 5"""
print("-- ยังไม่มี in prescription --")
display(pd.read_sql(q_no_rx, con))

## §5 RIGHT / FULL JOIN — SQLite ≥ 3.39 (สไลด์ 06)
- `RIGHT JOIN`: รักษาตารางขวาครบ (orphan rx จะเห็น!)
- `FULL OUTER JOIN`: ทั้งสองฝั่งครบ
- เวอร์ชันเก่า: emulate ด้วย LEFT สลับข้าง หรือ UNION


In [ ]:
# §5.1 RIGHT JOIN — เห็น orphan rx_id=6
if int(sqlite3.sqlite_version.split('.')[1]) >= 39 or int(sqlite3.sqlite_version.split('.')[0]) > 3:
    display(pd.read_sql("""
      SELECT r.rx_id, r.drug, p.hn
      FROM patients p
      RIGHT JOIN prescriptions r ON r.patient_id = p.patient_id
      ORDER BY r.rx_id""", con))
else:
    print("SQLite < 3.39 — emulate ด้วย LEFT สลับข้าง:")
    display(pd.read_sql("""
      SELECT r.rx_id, r.drug, p.hn
      FROM prescriptions r
      LEFT JOIN patients p ON r.patient_id = p.patient_id
      ORDER BY r.rx_id""", con))

# §5.2 FULL OUTER (ถ้ารองรับ)
try:
    n = cur.execute("SELECT COUNT(*) FROM patients p FULL OUTER JOIN prescriptions r ON r.patient_id=p.patient_id").fetchone()[0]
    print("FULL OUTER rows:", n)
except Exception as e:
    print("FULL ไม่รองรับ:", e)

## §6 SELF JOIN — ตารางเชื่อมตัวเอง (สไลด์ 07)
ต้องตั้ง alias 2 ชื่อ เช่น referral chain: doctor→doctor หรือ contact tracing

เราสร้าง `referrals(patient_id, referred_by)` เพื่อ demo


In [ ]:
cur.execute("DROP TABLE IF EXISTS referrals")
cur.execute("CREATE TABLE referrals(rx_from INTEGER, rx_to INTEGER)")
cur.executemany("INSERT INTO referrals VALUES (?,?)",
                [(1,2),(2,3),(3,1)])  # วนวงจรแนะนำตัว
con.commit()
# SELF JOIN: แปลง id → hn สองฝั่ง
q_self = """
SELECT a.rx_from, pa.hn AS from_hn, a.rx_to, pb.hn AS to_hn
FROM referrals a
JOIN patients pa ON pa.patient_id = a.rx_from
JOIN patients pb ON pb.patient_id = a.rx_to"""
display(pd.read_sql(q_self, con))
print("← alias a/b + pa/pb ชี้ตารางเดียวกัน 2 มุม (dot notation สไลด์ 08)")

## §7 UNION — ต่อผลลัพธ์แนวตั้ง (สไลด์ 08)
`UNION` ตัดซ้ำ · `UNION ALL` คงซ้ำ(เร็วกว่า) — จำนวน/ชื่อคอลัมน์ต้องตรง


In [ ]:
# §7 รวมรายชื่อคนในระบบ: มียา vs ไม่มียา
q_union = """
SELECT p.hn, 'มียา' AS status FROM patients p
 JOIN prescriptions r ON r.patient_id=p.patient_id
UNION
SELECT p.hn, 'ไม่มียา' FROM patients p
 LEFT JOIN prescriptions r ON r.patient_id=p.patient_id
 WHERE r.rx_id IS NULL
LIMIT 8"""
display(pd.read_sql(q_union, con))

# UNION ALL คง duplicate
n_u  = cur.execute("SELECT COUNT(*) FROM (SELECT hn FROM patients UNION SELECT hn FROM patients)").fetchone()[0]
n_ua = cur.execute("SELECT COUNT(*) FROM (SELECT hn FROM patients UNION ALL SELECT hn FROM patients)").fetchone()[0]
print(f"UNION={n_u} (ตัดซ้ำ) vs UNION ALL={n_ua}")

## §8 GROUP BY — สรุปรายกลุ่ม (สไลด์ 09)
Split-Apply-Combine: แบ่งกลุ่ม → aggregate → รวมผล · คอลัมน์ non-aggregate ต้องอยู่ใน GROUP BY


In [ ]:
# §8.1 ยอดจ่ายยาต่อผู้ป่วย (JOIN + GROUP BY + dot notation)
q_grp = """
SELECT p.hn, COUNT(r.rx_id) AS n_rx, SUM(r.cost_thb) AS total_cost, AVG(r.cost_thb) AS avg_cost
FROM patients p
LEFT JOIN prescriptions r ON r.patient_id = p.patient_id
GROUP BY p.patient_id
ORDER BY total_cost DESC NULLS LAST LIMIT 6"""
display(pd.read_sql(q_grp, con))

# §8.2 ยอดต่อชนิดยา
display(pd.read_sql("""
 SELECT drug, COUNT(*) AS n, SUM(cost_thb) AS spend
 FROM prescriptions GROUP BY drug ORDER BY spend DESC""", con))

# §8.3 ❌ error: non-aggregate ไม่อยู่ใน GROUP BY
try:
    pd.read_sql("SELECT hn, drug, COUNT(*) FROM prescriptions", con)
except Exception as e:
    print("Error (คาดไว้):", str(e)[:80])

## §9 HAVING — กรองหลังจัดกลุ่ม (สไลด์ 10)
WHERE กรอง**ก่อน** group · HAVING กรอง**หลัง** aggregate — เขียนผิดที่ = logic error


In [ ]:
# §9.1 เฉพาะผู้ป่วยที่จ่ายยาเกิน 1 รายการ
q_hav = """
SELECT p.hn, COUNT(r.rx_id) AS n_rx, SUM(r.cost_thb) AS total
FROM patients p LEFT JOIN prescriptions r ON r.patient_id=p.patient_id
GROUP BY p.patient_id
HAVING COUNT(r.rx_id) > 1
ORDER BY total DESC"""
display(pd.read_sql(q_hav, con))

# §9.2 WHERE+HAVING ร่วม: เฉพาะ Metformin, กลุ่มยาละเกิน 30 บาทเฉลี่ย
display(pd.read_sql("""
 SELECT drug, AVG(cost_thb) AS avg_cost, COUNT(*) AS n
 FROM prescriptions
 WHERE drug='Metformin'          -- ก่อน group
 GROUP BY drug
 HAVING AVG(cost_thb) > 30       -- หลัง group""", con))

## §10 Pipeline ฉบับเต็ม (สไลด์ 11)
`FROM(+JOIN) → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT` — ลอง query ที่ครบทุก clause


In [ ]:
q_full = """
SELECT p.gender, COUNT(DISTINCT p.patient_id) AS n_patients,
       SUM(r.cost_thb) AS spend
FROM patients p
LEFT JOIN prescriptions r ON r.patient_id=p.patient_id
WHERE p.birth_date LIKE '____-__-__'     -- ISO เท่านั้น
GROUP BY p.gender
HAVING SUM(r.cost_thb) > 0
ORDER BY spend DESC
LIMIT 5"""
display(pd.read_sql(q_full, con))
con.close(); print("closed ✅")

### ✅ Self-check (สไลด์ 12 Lab)
- INNER นับ rx ได้ 5 (orphan หล่น) · LEFT ได้ 6
- LEFT JOIN … IS NULL เจอผู้ป่วยไม่มียา
- GROUP BY drug ได้ 4 กลุ่ม · HAVING COUNT>1 ได้ 2 คน
- DELETE patient ที่มี rx โดย FK on → IntegrityError

### 📝 Homework 7
สร้างตาราง `visits(visit_id PK, patient_id FK, visit_date, hba1c)` + INSERT ≥5 แถว + query: ผู้ป่วยที่ hba1c เฉลี่ย>7 (JOIN+GROUP BY+HAVING) ส่ง .ipynb + .db

---
### 🔗 Specs รวม
[foreignkeys](https://www.sqlite.org/foreignkeys.html) · [JOIN](https://www.sqlite.org/lang_select.html#joinclause) · [UNION](https://www.sqlite.org/lang_select.html#compound) · [GROUP BY](https://www.sqlite.org/lang_select.html#groupby) · [HAVING](https://www.sqlite.org/lang_select.html#having) · [W3Schools JOINs](https://www.w3schools.com/sql/sql_join.asp) · [draw.io](https://www.drawio.com/)
